# 06 — Mid-Side EQ: Concepts & Practical Application

### Why this notebook exists

Notebook 01 used Mid/Side decomposition as a **measuring stick** — `Mid_dB − Side_dB` as a
proxy for "is the vocal loud enough?". This notebook is a *supplementary detour*. It looks at
the same Mid/Side idea from the other side of the engineer's desk: not measuring a mix, but
**changing** one.

Everything here is self-contained — the formulas, the code, and the audio players. You do not
need to have run any other notebook first (though it reads most naturally right after
**Notebook 01**, which introduces the L/R channels and the `Mid = (L+R)/2`, `Side = (L−R)/2`
trick this notebook builds on).

By the end, phrases like *"widen the air without brightening the vocal"* and *"tighten the low
end without collapsing the stereo field"* will be concrete sets of numbers you can measure on
this project's own tracks — and, more importantly, **hear**.

> **How you'll verify each idea:** every concept is paired with a real clip from this project's
> audio that you can press play on right here. Numbers serve your ears, not the other way
> around (per `AGENTS.md`).

## Setup

Same environment as the other notebooks: `librosa`, `numpy`, `matplotlib`,
`IPython.display.Audio` — plus `scipy.signal`, which is already part of the project's
environment and only appears in Section D. The one deliberate difference: we load Elissa's mix
in **stereo** (`mono=False`), because the entire point of this notebook is what happens
*between* the left and right channels.

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.signal import butter, lfilter, freqz, group_delay

DATA_DIR = "../data"
PLOTS_DIR = "../LABS/session_2026-07-31T120000/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

ELISSA_MIX = f"{DATA_DIR}/elisa_maktooba_leek.mp3"

def rms_db(x, eps=1e-10):
    '''Same loudness function as Notebook 00 — RMS energy converted to dB.'''
    rms = np.sqrt(np.mean(x ** 2))
    return 20 * np.log10(rms + eps)

def mid_side(y_stereo):
    '''Split a stereo array (shape 2 x n) into its Mid and Side channels.'''
    L, R = y_stereo[0], y_stereo[1]
    return (L + R) / 2.0, (L - R) / 2.0

def from_mid_side(mid, side):
    '''Recombine Mid and Side back into a stereo array (shape 2 x n).'''
    return np.stack([mid + side, mid - side])

# One short clip, reused across every demo. Change CLIP_OFFSET to hear another moment.
CLIP_OFFSET = 45.0
CLIP_SEC = 8.0

y_clip, sr = librosa.load(ELISSA_MIX, sr=22050, mono=False,
                          offset=CLIP_OFFSET, duration=CLIP_SEC)
mid, side = mid_side(y_clip)

print("Setup complete. Elissa stereo clip loaded: "
      f"{CLIP_SEC}s starting at {CLIP_OFFSET}s, sr={sr} Hz")

## A. Mid/Side as a signal flow, not just a measurement

Notebook 01 introduced the two formulas:

$$ \text{Mid} = \frac{L + R}{2} \qquad \text{Side} = \frac{L - R}{2} $$

along with the property that makes them useful: **it's reversible** — `L = Mid + Side`,
`R = Mid − Side`. Engineers don't just *measure* with this. They route the signal through it,
process Mid and Side with *different* EQs, and route back into L/R. The signal flow is:

```
L/R ──▶  M = (L+R)/2  ──▶  [EQ Mid]  ──▶┐
         S = (L−R)/2  ──▶  [EQ Side] ──▶┴──▶  L' = M'+S'   R' = M'−S'  ──▶ L/R
```

Before we trust anything that happens inside that box, let's prove the box itself is lossless:
encode a clip into Mid/Side and immediately decode it back. If the round trip is exact, then any
change you hear later in this notebook is caused by the EQs we insert — not by the transform.

In [ ]:
# Encode -> decode immediately, with nothing in between. Should be sample-perfect.
y_back = from_mid_side(*mid_side(y_clip))

max_err = np.max(np.abs(y_clip - y_back))
print(f"Max |original − round-trip| = {max_err:.3e}")
if max_err < 1e-12:
    print("=> Lossless. Every change later in this notebook is caused by our EQs,")
    print("   not by the Mid/Side transform itself.")
else:
    print("=> WARNING: the transform is NOT lossless. Stop and investigate.")

print("\nOriginal:")
display(Audio(y_clip, rate=sr))
print("Round-tripped through Mid/Side:")
display(Audio(y_back, rate=sr))

**Look at the two players above — or rather, don't be able to tell them apart.** The printed
error is on the order of 10⁻¹⁶, the floating-point noise floor: the round trip is sample-perfect.
This is the foundation of everything that follows — *whatever you hear next is the EQ, not the
math.*

## B. Why EQ Mid and Side separately?

A normal stereo EQ applies the *same* curve to both L and R. That's a blunt instrument: you
cannot touch the centered vocal without also touching the panned reverb tails that share the same
frequencies.

Mid/Side EQ separates those two problems cleanly:

- **Mid** carries everything panned to the center — lead vocal, kick, bass, snare. The "body" of
  the mix.
- **Side** carries everything that is *different* between the channels — panned instruments,
  stereo reverb, width effects. The "space" of the mix.

So engineers make moves like *"cut the mud from the center without collapsing the stereo field"*
or *"add air to the edges without brightening the vocal."* Each of those is impossible with a
plain L/R EQ and trivial with Mid/Side.

The helper `eq_fft` below is a tiny EQ: it applies a straight-line (in log-frequency) dB curve to
a single channel. Feeding it the Side channel and recombining reproduces exactly what a real
Mid/Side EQ does. Let's make three of the most common moves and listen.

In [ ]:
def eq_fft(x, sr, gains, taper_sec=0.05):
    '''Apply a piecewise-linear (in log-frequency) dB gain curve in the frequency domain.

    gains : list of (freq_Hz, gain_dB) points defining the curve, sorted by frequency,
            e.g. [(20, 0), (2500, 0), (3500, -30), (20000, -30)]  # steep high cut
    '''
    n = len(x)
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(n, d=1.0 / sr)
    logf = np.log2(np.maximum(freqs, 1e-9))
    f_pts = np.array([g[0] for g in gains])
    db_pts = np.array([g[1] for g in gains])
    gain_db = np.interp(logf, np.log2(f_pts), db_pts)
    X = X * (10 ** (gain_db / 20.0))
    y = np.fft.irfft(X, n)

    # Crossfade the edges back to the original so the clip starts/stops cleanly
    # (the FFT assumes the signal repeats; this hides that circular-wrap artifact).
    nt = int(taper_sec * sr)
    ramp = np.linspace(0, 1, nt)
    y[:nt]  = x[:nt]  * (1 - ramp) + y[:nt]  * ramp
    y[-nt:] = x[-nt:] * (1 - ramp[::-1]) + y[-nt:] * ramp[::-1]
    return y

def process_side(y_stereo, gains):
    '''EQ only the Side channel, then recombine — a classic Mid/Side move.'''
    m, s = mid_side(y_stereo)
    return from_mid_side(m, eq_fft(s, sr, gains))

# Move 1 — "Narrow": cut everything above ~3 kHz in the Side.
y_narrow = process_side(y_clip, [(20, 0), (2500, 0), (3500, -30), (20000, -30)])

# Move 2 — "Widen": add air (5-12 kHz) to the Side only.
y_wide = process_side(y_clip, [(20, 0), (5000, 0), (12000, 6), (20000, 6)])

# Move 3 — "Clean low-end": high-pass the Side below ~120 Hz.
y_clean_low = process_side(y_clip, [(20, -30), (100, -30), (140, 0), (20000, 0)])

print("ORIGINAL:")
display(Audio(y_clip, rate=sr))
print("1) NARROW  — Side high-cut above ~3 kHz:")
display(Audio(y_narrow, rate=sr))
print("2) WIDE    — Side air boost 5-12 kHz:")
display(Audio(y_wide, rate=sr))
print("3) CLEAN LOW-END — Side high-pass below ~120 Hz:")
display(Audio(y_clean_low, rate=sr))

**What to listen for, in order:**

1. **Narrow** — the image should visibly pull toward the center: reverb and pads lose their
   spread while the vocal (centered, living in Mid) keeps its volume. This is the move a "mono
   button" — or a stereo *widener in reverse* — approximates.
2. **Wide** — the mix opens up around the edges — air, cymbals, reverb shine — while the vocal's
   tone barely changes, because we never touched Mid.
3. **Clean low-end** — the subtlest of the three, by design. Most mixes have almost no bass in
   the Side channel (bass is kept centered for mono compatibility). The move's job is to *keep it
   that way* — to kill whatever low rumble and phasey mud does leak into the sides.

The center stays put in all three. **That is the entire point of Mid/Side EQ: you can reshape the
width and the edges without touching the middle, and vice versa.**

## C. Common Mid-Side EQ moves (and how to prove them)

Here's the condensed version of what mixing/mastering engineers actually do with Mid/Side EQ:

| Goal | The move | Why it works |
|---|---|---|
| Tighter, mono-safe bass | High-pass the **Side** below ~120 Hz | Panned sub bass phase-cancels in mono; centered bass is always safe |
| Cleaner, wider low-mids | Cut 250–400 Hz in the **Side** | Low-mid mud pools in the edges; cutting it there widens the image instead of thinning the center |
| "Air" without a harsh vocal | Boost 8–16 kHz in the **Side** only | Air lives at the edges; the centered vocal keeps its presence band untouched |
| De-ess a centered vocal | Cut ~5–8 kHz in the **Mid** only | Sibilance is center-panned; the cymbal sparkle in the sides is preserved |
| Punchier drums, not louder vocals | Cut 200–400 Hz in the **Mid** only | The low-mid "boom" of the kick/snare is centered content |

The claim hiding in that table is that these moves change *where energy sits* in Mid vs Side. So
let's measure it. We'll apply the **de-ess** move (cut 5–8 kHz in the Mid channel only), then
plot the average spectrum of Mid *and* of Side, before and after. If the theory is right, only
the Mid spectrum should drop inside 5–8 kHz.

In [ ]:
def spectral_profile(x, sr=sr, n_fft=2048):
    '''Median magnitude spectrum of a mono signal, in dB.'''
    S = np.abs(librosa.stft(x, n_fft=n_fft))
    return librosa.fft_frequencies(sr=sr, n_fft=n_fft), \
           20 * np.log10(np.median(S, axis=1) + 1e-12)

def process_mid(y_stereo, gains):
    '''EQ only the Mid channel, then recombine.'''
    m, s = mid_side(y_stereo)
    return from_mid_side(eq_fft(m, sr, gains), s)

# The de-ess move: cut 5-8 kHz in the Mid channel only.
y_deessed = process_mid(y_clip, [(20, 0), (5000, 0), (8000, -6), (20000, -6)])

freqs, mid_before  = spectral_profile(mid)
_,     mid_after   = spectral_profile(mid_side(y_deessed)[0])
_,     side_before = spectral_profile(side)
_,     side_after  = spectral_profile(mid_side(y_deessed)[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, title, b, a_ in [
        (axes[0], "MID channel — before vs after", mid_before, mid_after),
        (axes[1], "SIDE channel — before vs after", side_before, side_after)]:
    ax.semilogx(freqs, b, label="before", color="steelblue")
    ax.semilogx(freqs, a_, label="after", color="crimson")
    ax.axvspan(5000, 8000, color="gold", alpha=0.15)
    ax.set_title(title)
    ax.set_xlim(20, 10000)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude (dB)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/mid_side_eq_spectral_proof.png", dpi=110)
plt.show()

print("De-essed (Mid-only 5-8 kHz cut) — the vocal should sound a touch less sibilant:")
display(Audio(y_deessed, rate=sr))

**Look at the gold band (5–8 kHz).** In the *Mid* panel the red curve dips below the blue one —
we cut exactly what we asked for. In the *Side* panel the two curves sit essentially on top of
each other — the sides were untouched, even though they contain the same frequencies. One move,
one channel, and the plot shows precisely that. This is the whole mechanism of Mid/Side EQ made
visible as numbers.

And in the de-essed clip: the vocal's *"s"* and *"sh"* sounds should be slightly softer, while
the cymbals and air at the edges keep their brightness.

## D. The phase trap: why "just EQ it" isn't always safe

Mid/Side EQ has a catch — and it's the reason mastering plugins offer a **linear-phase** toggle.

An ordinary (minimum-phase) EQ does *two* things at once: it changes volume per frequency (what
you asked for) *and* it delays different frequencies by different amounts (a side effect you
didn't ask for). This frequency-dependent delay is called **group delay**.

In a plain stereo EQ that's usually harmless: L and R get the same delay, so the stereo image is
unaffected. But the moment you apply **different** EQs to Mid and Side, those delays get baked
into the recombined L/R. The phase relationship between "center" (Mid) and "edges" (Side)
changes, and the image can smear — transients go soft, and the mix can feel like it wobbles on
wide material.

The fix is a **linear-phase (or zero-phase) EQ** — one that delays every frequency equally, so
the only thing it changes is volume. Let's demonstrate, using a matched pair: the *identical*
magnitude curve (a low-pass at 500 Hz) applied to the Side channel once minimum-phase and once
zero-phase. Because the magnitudes match by construction, **any audible difference is purely the
phase.**

In [ ]:
def butter_min_phase(x, cut_hz, sr, order=4):
    '''Ordinary Butterworth low-pass: changes volume AND delays frequencies unequally.'''
    b, a = butter(order, cut_hz / (sr / 2))
    return lfilter(b, a, x)

def butter_zero_phase(x, cut_hz, sr, order=4):
    '''Same magnitude curve, but zero phase: only changes volume.'''
    b, a = butter(order, cut_hz / (sr / 2))
    n = len(x)
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(n, d=1.0 / sr)
    _, h = freqz(b, a, worN=len(freqs))
    return np.fft.irfft(X * np.abs(h), n)

CUT_HZ = 500.0

# Same low-pass applied to the Side channel only, two different phase behaviors:
m, s = mid_side(y_clip)
y_min = from_mid_side(m, butter_min_phase(s, CUT_HZ, sr))
y_lin = from_mid_side(m, butter_zero_phase(s, CUT_HZ, sr))

# Because the magnitude curves are identical, this difference is PURE phase:
phase_only = y_min - y_lin
print(f"Side-only low-pass @ {CUT_HZ:.0f} Hz")
print(f"  level of phase-only difference : {rms_db(phase_only):+.2f} dB")
print(f"  level of the original clip     : {rms_db(y_clip):+.2f} dB")
print("  -> if phase were irrelevant, the first number would be around -infinity.")

# Sanity check: the SAME filter on BOTH Mid and Side == filtering L and R directly,
# which is why plain stereo EQ never has this problem.
b, a = butter(4, CUT_HZ / (sr / 2))
L, R = y_clip[0], y_clip[1]
direct = np.stack([lfilter(b, a, L), lfilter(b, a, R)])
via_ms = from_mid_side(lfilter(b, a, m), lfilter(b, a, s))
print(f"  same filter on both M/S == direct L/R EQ?  "
      f"max error {np.max(np.abs(via_ms - direct)):.2e}")

# Group delay: minimum-phase delays frequencies unequally; zero-phase delays none.
w = np.linspace(0, np.pi, 1024, endpoint=False)
gd_w, gd = group_delay((b, a), w=w)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gd_w * sr / (2 * np.pi), gd, label="minimum-phase EQ", color="crimson")
ax.axhline(0, color="steelblue", linestyle="--", label="zero-phase EQ")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Group delay (samples)")
ax.set_title(f"Group delay of a {CUT_HZ:.0f} Hz low-pass on the Side channel")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/group_delay_min_vs_zero_phase.png", dpi=110)
plt.show()

print("Minimum-phase version (transients may smear):")
display(Audio(y_min, rate=sr))
print("Zero-phase version (same volume change, no phase smear):")
display(Audio(y_lin, rate=sr))

**The number and the picture agree.** The phase-only difference sits only ~11 dB below the
original clip — that is not numerical noise, it's an audible signal. And the group-delay plot
shows why: the minimum-phase filter delays the Side content by up to ~27 samples near 500 Hz (and
nearly zero far from it), while the zero-phase version delays nothing. When the delayed Side is
recombined with the un-delayed Mid, the two no longer line up in time the way they did in the
original mix.

A 4-second clip with a 500 Hz low-pass on the Side is a deliberately *harsh* example so the
effect is easy to hear — a real Mid/Side EQ in mastering uses gentler curves. But the mechanism is
the same at any depth, which is exactly why serious Mid/Side tools ship with a linear-phase
option.

## E. Seeing the width: the goniometer

One more visualization engineers reach for constantly: the **goniometer** (a.k.a. Lissajous /
vectorscope). Plot every sample of L on the x-axis and R on the y-axis:

- A **mono** signal has `L == R`, so every point lands on the diagonal line `y = x`.
- A **wide** signal has lots of `L ≠ R` content, so the points spray out into a cloud.

The cloud's spread roughly equals the Side content (width); the density along the diagonal equals
the Mid content (center). Let's draw goniometers for the original clip and for the NARROW and WIDE
versions from Section B — then compute one number that summarizes what you see: Side energy
relative to Mid energy.

In [ ]:
def goniometer(ax, y_stereo, title, downsample=64):
    '''Scatter-plot L vs R (downsampled for speed/readability).'''
    Lg, Rg = y_stereo[0][::downsample], y_stereo[1][::downsample]
    ax.scatter(Lg, Rg, s=1, alpha=0.05, linewidths=0)
    lim = max(np.max(np.abs(Lg)), np.max(np.abs(Rg))) * 1.1
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.plot([-lim, lim], [-lim, lim], color="gray", linestyle="--",
            linewidth=0.8, label="mono diagonal")
    ax.set_title(title)
    ax.set_xlabel("Left")
    ax.set_ylabel("Right")
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
goniometer(axes[0], y_clip,   "ORIGINAL")
goniometer(axes[1], y_narrow, "NARROW  (Side high-cut)")
goniometer(axes[2], y_wide,   "WIDE    (Side air boost)")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/goniometer_compare.png", dpi=110)
plt.show()

def width_metric(y_stereo):
    '''Side RMS relative to Mid RMS, in dB. Higher = wider stereo image.'''
    m, s = mid_side(y_stereo)
    return rms_db(s) - rms_db(m)

print("Width (Side_dB − Mid_dB):")
print(f"  ORIGINAL : {width_metric(y_clip):+6.2f} dB")
print(f"  NARROW   : {width_metric(y_narrow):+6.2f} dB")
print(f"  WIDE     : {width_metric(y_wide):+6.2f} dB")

**Compare the three clouds.** The NARROW plot hugs the gray diagonal — the mix has pulled toward
the center. The WIDE plot sprays furthest off the diagonal — the edges have grown. The ORIGINAL
sits between the two. The printed width numbers (`Side_dB − Mid_dB`) say the same thing as a
single figure: higher = wider.

You now have the full toolkit in miniature: the lossless transform (A), the moves (B), the proof
(C), the warning (D), and the meter (E).

## Self-Check

Try to answer each question yourself (out loud or on paper) *before* expanding the answer. If you
get one wrong, re-read that section — don't just move on.

<details>
<summary><b>Q1.</b> In your own words: why can't a normal (L/R) stereo EQ "de-ess the vocal without touching the cymbals" the way a Mid/Side EQ can?</summary>

**A1.** A normal stereo EQ applies the same gain curve to both channels, so any change hits L and R identically — and the vocal's sibilance and the cymbals' brightness live at overlapping frequencies. In Mid/Side, the vocal is (mostly) centered content living in the Mid channel, while the panned cymbals and their stereo spread live in the Side channel. Cutting 5–8 kHz in the Mid channel reduces only the sibilance; the Side (and its cymbals) is untouched.
</details>

<details>
<summary><b>Q2.</b> You want to add "air" (8–16 kHz) to a mix without making the lead vocal harsh. Which Mid/Side move do you make, and why does it leave the vocal alone?</summary>

**A2.** Boost 8–16 kHz in the **Side** channel only. The lead vocal is centered, so it lives in Mid. Boosting only the Side raises the level of the wide/ambient material (reverb, cymbals, room shine) while the Mid channel — and therefore the vocal — is unchanged.
</details>

<details>
<summary><b>Q3.</b> This notebook applied the identical magnitude curve to the Side channel twice — once with a minimum-phase filter and once with a zero-phase filter — and the two results differed audibly. What exactly differed, and why does it matter specifically for Mid/Side EQ?</summary>

**A3.** Only the phase behavior differed: the minimum-phase filter delays different frequencies by different amounts (group delay), while the zero-phase filter delays everything equally. In a Mid/Side chain, that uneven delay shifts the time relationship between the Mid and Side components once they are recombined into L/R, smearing transients and subtly distorting the stereo image. Plain L/R EQ avoids this because both channels get the same delay; Mid/Side EQ applies different filters per channel, so phase becomes audible — which is why serious Mid/Side EQs offer a linear-phase mode.
</details>

<details>
<summary><b>Q4.</b> A goniometer (L-vs-R scatter) of a track shows a thin cloud hugging the diagonal. What does that tell you about the mix, and what single number from this notebook quantifies the same thing?</summary>

**A4.** Points on the diagonal mean L ≈ R, i.e. mostly mono/centered content — the mix is narrow. The single number that quantifies it is the width metric, `Side_dB − Mid_dB`: smaller (more negative) means narrower; larger means wider.
</details>

## Summary — what you now have

- **Mid/Side is a lossless signal flow** — encode, process each channel independently, re-encode —
  verified sample-perfect on this project's audio.
- **Mid/Side EQ separates "center" from "edges"** — moves like "widen the air without brightening
  the vocal," "tighten the low end without collapsing width," and "de-ess the vocal without
  dulling the cymbals" are only possible when Mid and Side are EQ'd independently.
- **The moves are provable** — cutting 5–8 kHz in the Mid channel changed exactly that band in the
  Mid spectrum and left the Side spectrum untouched.
- **The catch is phase** — minimum-phase EQs delay frequencies unequally; applied to Mid and Side
  separately, that delay smears the recombined image. Linear/zero-phase EQs avoid it.
- **The meter** — a goniometer shows width visually; `Side_dB − Mid_dB` is the same idea as a
  single number.

This was a **supplementary** notebook: it doesn't advance the main curriculum (Notebooks 00–05),
it deepens a concept those notebooks keep leaning on. The reason Notebook 01's proxy worked as a
*measurement* is the same reason engineers reach for Mid/Side as a *tool*: the center of a mix and
the edges of a mix carry different things, and once you can separate them you can measure them
separately — and sculpt them separately.